In [2]:
import pandas as pd
import random
import numpy as np

n = 2000

jobs = [
    "Machine Learning Engineer",
    "Data Scientist",
    "AI Researcher",
    "Business Intelligence Analyst",
    "Data Analyst",
    "Software Engineer",
    "Backend Developer",
    "Frontend Developer",
    "Cloud Engineer",
    "DevOps Engineer",
    "Cybersecurity Analyst",
    "Security Engineer",
    "Full Stack Developer"
]

def score_jobs(profile):
    scores = {job: 0 for job in jobs}

    # ML Profile
    if profile["python"] and profile["machine_learning"]:
        scores["Machine Learning Engineer"] += 3
        scores["Data Scientist"] += 2
        scores["AI Researcher"] += 2

    # Data profile
    if profile["sql"] and profile["data_analysis"]:
        scores["Business Intelligence Analyst"] += 3
        scores["Data Analyst"] += 2
        scores["Data Scientist"] += 1

    # Software profile
    if profile["java"] or profile["c_cpp"]:
        scores["Software Engineer"] += 3
        scores["Backend Developer"] += 2

    # Cloud / DevOps
    if profile["cloud_computing"]:
        scores["Cloud Engineer"] += 3
    if profile["devops"]:
        scores["DevOps Engineer"] += 3

    # Cyber
    if profile["cybersecurity"]:
        scores["Cybersecurity Analyst"] += 3
    if profile["networking"]:
        scores["Security Engineer"] += 2

    # Web
    if profile["web_development"]:
        scores["Frontend Developer"] += 2
        scores["Full Stack Developer"] += 2

    # Soft skill influence
    if profile["communication"] >= 4:
        scores["Business Intelligence Analyst"] += 1

    # GPA influence
    if profile["gpa"] > 3.7:
        scores["AI Researcher"] += 2

    return scores

data = []

for i in range(n):
    profile = {
        "age": random.randint(21, 35),
        "gender": random.choice(["Male", "Female"]),
        "degree_level": random.choice(["Bachelor", "Master", "PhD"]),
        "field_of_study": random.choice(["Computer Science","Data Science","AI","Cybersecurity","Software Engineering"]),
        "gpa": round(random.uniform(2.5,4.0),2),
        "years_experience": random.randint(0,8),
        "python": random.randint(0,1),
        "java": random.randint(0,1),
        "c_cpp": random.randint(0,1),
        "sql": random.randint(0,1),
        "machine_learning": random.randint(0,1),
        "data_analysis": random.randint(0,1),
        "cloud_computing": random.randint(0,1),
        "cybersecurity": random.randint(0,1),
        "web_development": random.randint(0,1),
        "devops": random.randint(0,1),
        "networking": random.randint(0,1),
        "communication": random.randint(1,5),
        "leadership": random.randint(1,5),
        "problem_solving": random.randint(1,5),
        "teamwork": random.randint(1,5),
        "adaptability": random.randint(1,5)
    }

    scores = score_jobs(profile)

    # Add 10% noise
    if random.random() < 0.1:
        random_job = random.choice(jobs)
        scores[random_job] += 2

    top3 = sorted(scores, key=scores.get, reverse=True)[:3]

    row = list(profile.values()) + top3
    data.append(row)

columns = list(profile.keys()) + ["recommended_job_1","recommended_job_2","recommended_job_3"]

df = pd.DataFrame(data, columns=columns)
df.to_csv("career_multilabel_dataset.csv", index=False)

print("Dataset generated successfully!")

Dataset generated successfully!


In [3]:
import pandas as pd
import numpy as np

# ==============================
# 1️⃣ Load Dataset
# ==============================
df = pd.read_csv("career_multilabel_dataset.csv")

print("Initial shape:", df.shape)

# ==============================
# 2️⃣ Remove Duplicates
# ==============================
df.drop_duplicates(inplace=True)
print("After removing duplicates:", df.shape)

# ==============================
# 3️⃣ Handle Missing Values
# ==============================

# Check missing
print("\nMissing values:\n", df.isnull().sum())

# Numeric columns → fill with median
numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Categorical columns → fill with mode
categorical_cols = df.select_dtypes(include="object").columns
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# ==============================
# 4️⃣ Fix Data Types
# ==============================

# Convert binary skill columns to int
binary_cols = [
    "python","java","c_cpp","sql","machine_learning",
    "data_analysis","cloud_computing","cybersecurity",
    "web_development","devops","networking"
]

for col in binary_cols:
    df[col] = df[col].astype(int)

# Convert soft skills to int
soft_skills = ["communication","leadership","problem_solving","teamwork","adaptability"]
for col in soft_skills:
    df[col] = df[col].astype(int)

# ==============================
# 5️⃣ Outlier Handling
# ==============================

# Clip unrealistic values
df["age"] = df["age"].clip(18, 60)
df["gpa"] = df["gpa"].clip(0, 4)
df["years_experience"] = df["years_experience"].clip(0, 40)

# ==============================
# 6️⃣ Normalize Text Columns
# ==============================

text_cols = ["degree_level","field_of_study","gender",
             "recommended_job_1","recommended_job_2","recommended_job_3"]

for col in text_cols:
    df[col] = df[col].str.strip().str.title()

# ==============================
# 7️⃣ Ensure Multi-label Columns Are Unique
# ==============================

def ensure_unique_jobs(row):
    jobs = [row["recommended_job_1"],
            row["recommended_job_2"],
            row["recommended_job_3"]]
    unique_jobs = list(dict.fromkeys(jobs))

    while len(unique_jobs) < 3:
        unique_jobs.append("Unknown")

    row["recommended_job_1"], row["recommended_job_2"], row["recommended_job_3"] = unique_jobs[:3]
    return row

df = df.apply(ensure_unique_jobs, axis=1)

# ==============================
# 8️⃣ Optional: Create Combined Target Column
# ==============================

df["recommended_jobs"] = (
    df["recommended_job_1"] + "," +
    df["recommended_job_2"] + "," +
    df["recommended_job_3"]
)

# ==============================
# 9️⃣ Save Cleaned Dataset
# ==============================

df.to_csv("career_multilabel_dataset_cleaned.csv", index=False)

print("\nCleaning completed.")
print("Final shape:", df.shape)

Initial shape: (2000, 25)
After removing duplicates: (2000, 25)

Missing values:
 age                  0
gender               0
degree_level         0
field_of_study       0
gpa                  0
years_experience     0
python               0
java                 0
c_cpp                0
sql                  0
machine_learning     0
data_analysis        0
cloud_computing      0
cybersecurity        0
web_development      0
devops               0
networking           0
communication        0
leadership           0
problem_solving      0
teamwork             0
adaptability         0
recommended_job_1    0
recommended_job_2    0
recommended_job_3    0
dtype: int64

Cleaning completed.
Final shape: (2000, 26)


In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import hamming_loss, f1_score, classification_report

# 1️⃣ Load cleaned dataset
df = pd.read_csv("career_multilabel_dataset_cleaned.csv")

# 2️⃣ Prepare multi-label target
df["job_list"] = df[[
    "recommended_job_1",
    "recommended_job_2",
    "recommended_job_3"
]].values.tolist()

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df["job_list"])

# 3️⃣ Prepare features
X = df.drop(columns=[
    "recommended_job_1",
    "recommended_job_2",
    "recommended_job_3",
    "recommended_jobs",
    "job_list"
], errors="ignore")
X = pd.get_dummies(X, drop_first=True)

# 4️⃣ Train-test split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)


In [10]:
# 5️⃣ Train OneVsRest + Random Forest with balanced class weights
model = OneVsRestClassifier(
    RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        class_weight='balanced',  # ✅ fixes rare class issue
        random_state=42,
        n_jobs=-1
    )
)

model.fit(X_train, Y_train)
print("Model training completed ✅")

# 6️⃣ Predictions
Y_pred = model.predict(X_test)
Y_prob = model.predict_proba(X_test)

# 7️⃣ Evaluation
print("\nHamming Loss:", hamming_loss(Y_test, Y_pred))
print("Micro F1 Score:", f1_score(Y_test, Y_pred, average="micro"))
print("Macro F1 Score:", f1_score(Y_test, Y_pred, average="macro"))
print("\nClassification Report:\n")
print(classification_report(Y_test, Y_pred, target_names=mlb.classes_))

# 8️⃣ Precision(Top-3 recommendation)
def precision_at_k(y_true, y_prob, k=3):
    correct = 0
    total = y_true.shape[0]

    for i in range(total):
        top_k_idx = np.argsort(y_prob[i])[::-1][:k]
        true_labels = np.where(y_true[i]==1)[0]
        correct += len(set(top_k_idx) & set(true_labels)) / k
    return correct / total

p3 = precision_at_k(Y_test, Y_prob, k=3)
print("\nPrecision@3:", round(p3, 3))

Model training completed ✅

Hamming Loss: 0.04346153846153846
Micro F1 Score: 0.8989266547406082
Macro F1 Score: 0.6208661808099218

Classification Report:

                               precision    recall  f1-score   support

                Ai Researcher       0.97      0.62      0.76        58
            Backend Developer       0.98      0.47      0.64        87
Business Intelligence Analyst       0.99      0.89      0.94        92
               Cloud Engineer       0.95      1.00      0.97       174
        Cybersecurity Analyst       0.98      0.94      0.96       115
                 Data Analyst       1.00      0.08      0.14        13
               Data Scientist       1.00      0.46      0.63        39
              Devops Engineer       0.93      0.97      0.95       153
           Frontend Developer       1.00      0.09      0.17        32
         Full Stack Developer       0.00      0.00      0.00         6
    Machine Learning Engineer       1.00      0.86      0.92 

c:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [8]:
from sklearn.multioutput import ClassifierChain
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# ============================================================
# 5️⃣ Entraînement avec ClassifierChain (Ordre Aléatoire)
# ============================================================

# On définit le modèle de base (Random Forest)
base_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Configuration du ClassifierChain
# order='random' : Les chaînes sont construites dans un ordre aléatoire des labels
# random_state=42 : Pour garantir la reproductibilité des résultats
chain_model = ClassifierChain(base_rf, order='random', random_state=42)

# Entraînement du modèle
chain_model.fit(X_train, Y_train)
print("Entraînement du ClassifierChain terminé ✅")

# ============================================================
# 6️⃣ Prédictions
# ============================================================

# Prédictions binaires (0 ou 1)
Y_pred_chain = chain_model.predict(X_test)

# Prédictions des probabilités (nécessaire pour Precision@K)
# Note : ClassifierChain renvoie les probas sous forme (n_labels, n_samples).T
Y_prob_chain = chain_model.predict_proba(X_test)

# ============================================================
# 7️⃣ Évaluation des performances
# ============================================================

print("\n--- Résultats ClassifierChain ---")
print("Hamming Loss :", hamming_loss(Y_test, Y_pred_chain))
print("Micro F1 Score:", f1_score(Y_test, Y_pred_chain, average="micro"))
print("Macro F1 Score:", f1_score(Y_test, Y_pred_chain, average="macro"))

print("\nClassification Report:\n")
print(classification_report(Y_test, Y_pred_chain, target_names=mlb.classes_))

# ============================================================
# 8️⃣ Precision@3 (Top-3 recommandations)
# ============================================================

# Utilisation de votre fonction existante pour comparer
p3_chain = precision_at_k(Y_test, Y_prob_chain, k=3)

print(f"\nPrecision@3 (Chain): {round(p3_chain, 3)}")

Entraînement du ClassifierChain terminé ✅

--- Résultats ClassifierChain ---
Hamming Loss : 0.03480769230769231
Micro F1 Score: 0.9197339246119733
Macro F1 Score: 0.6441546297951148

Classification Report:

                               precision    recall  f1-score   support

                Ai Researcher       0.97      0.57      0.72        58
            Backend Developer       0.97      0.67      0.79        87
Business Intelligence Analyst       0.99      0.88      0.93        92
               Cloud Engineer       0.98      1.00      0.99       174
        Cybersecurity Analyst       0.98      0.97      0.98       115
                 Data Analyst       1.00      0.08      0.14        13
               Data Scientist       0.97      0.77      0.86        39
              Devops Engineer       0.98      0.99      0.98       153
           Frontend Developer       1.00      0.03      0.06        32
         Full Stack Developer       0.00      0.00      0.00         6
    Machine

c:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [11]:
from sklearn.multioutput import ClassifierChain
import numpy as np

# ============================================================
# 5️⃣ Évaluation : ClassifierChain (Ordre par défaut)
# ============================================================

# On réutilise le même modèle de base pour une comparaison équitable
base_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Configuration du ClassifierChain avec l'ordre par défaut
# order=None : Le modèle suit l'ordre naturel des colonnes de Y_train
chain_default = ClassifierChain(base_rf, order=None, random_state=42)

# Entraînement
print("Entraînement du ClassifierChain (ordre par défaut) en cours...")
chain_default.fit(X_train, Y_train)

# ============================================================
# 6️⃣ Prédictions et Probabilités
# ============================================================

# Prédictions binaires
Y_pred_default = chain_default.predict(X_test)

# Prédictions des probabilités pour le Precision@3
# On transpose (.T) pour que le format soit compatible avec le calcul de précision
Y_prob_default = chain_default.predict_proba(X_test)

# ============================================================
# 7️⃣ Analyse des résultats
# ============================================================

print(f"\n{'='*50}")
print("🔹 RÉSULTATS : CLASSIFIER CHAIN (DEFAULT ORDER)")
print(f"{'='*50}")

# Calcul des métriques
hl_def   = hamming_loss(Y_test, Y_pred_default)
mif1_def = f1_score(Y_test, Y_pred_default, average='micro')
p3_def   = precision_at_k(Y_test, Y_prob_default, k=3)

print(f"Hamming Loss (Erreur globale) : {hl_def:.4f}")
print(f"Micro F1 Score                : {mif1_def:.4f}")
print(f"Precision@3 (Top 3 métiers)   : {p3_def:.4f}")

# Rapport détaillé par métier
print("\nClassification Report détaillé :")
print(classification_report(Y_test, Y_pred_default, target_names=mlb.classes_))

Entraînement du ClassifierChain (ordre par défaut) en cours...

🔹 RÉSULTATS : CLASSIFIER CHAIN (DEFAULT ORDER)
Hamming Loss (Erreur globale) : 0.0350
Micro F1 Score                : 0.9187
Precision@3 (Top 3 métiers)   : 0.9592

Classification Report détaillé :
                               precision    recall  f1-score   support

                Ai Researcher       0.97      0.62      0.76        58
            Backend Developer       0.98      0.56      0.72        87
Business Intelligence Analyst       0.99      0.88      0.93        92
               Cloud Engineer       0.98      1.00      0.99       174
        Cybersecurity Analyst       0.98      0.97      0.98       115
                 Data Analyst       1.00      0.23      0.38        13
               Data Scientist       1.00      0.62      0.76        39
              Devops Engineer       1.00      0.93      0.97       153
           Frontend Developer       1.00      0.34      0.51        32
         Full Stack Develop

c:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [12]:
exact_match_acc = np.mean(np.all(Y_test == Y_pred, axis=1))
print("Exact-match accuracy:", round(exact_match_acc, 3))

Exact-match accuracy: 0.532
